In [3]:
import os
from typesafe_sdk import TypeSafeClient, Choice, Noul, Score

os.environ["TYPESAFE_BASE_URL"] = "http://localhost:11434"
os.environ["TYPESAFE_API_KEY"] = "ollama"
os.environ["TYPESAFE_DEFAULT_MODEL"] = "nimble"

## Formato del Contexto en el state
Puedes estructurar el state pasando un texto en formato clave-valor o un diccionario Python / JSON, ya que nimble lo interpreta limpiamente:

In [1]:
# Definir el contexto previo y la entrada actual
contexto_sistema = """
POLÍTICA DE DEVOLUCIONES:
- Solo se aceptan reembolsos si la compra se realizó en los últimos 30 días.
- Productos en oferta o liquidados NO tienen reembolso.
- El usuario debe presentar número de factura.
"""

consulta_usuario = "Compré una camisa en oferta hace 10 días y quiero que me devuelvan el dinero. Factura #8821."

# Estado estructurado combinando Contexto + Entrada
state_data = {
    "contexto_politica": contexto_sistema,
    "consulta_cliente": consulta_usuario
}


## Definición de Preguntas e Instrucciones
Define las preguntas apuntando a evaluar la coincidencia entre la consulta y el contexto:

In [4]:
from typesafe_sdk import TypeSafeClient, Choice, Noul

questions = {
    "cumple_politica_tiempo": Noul(
        instructions="Según el contexto de la política, ¿la solicitud está dentro del plazo permitido de 30 días?"
    ),
    "es_producto_elegible": Noul(
        instructions="Según el contexto de la política, ¿el producto mencionado es elegible para reembolso?"
    ),
    "decision_final": Choice(
        instructions="¿Cuál es la decisión que debe tomar el sistema según las políticas expuestas en el contexto?",
        criteria={
            "aprobar": "La compra cumple todos los requisitos (tiempo, elegibilidad y factura).",
            "rechazar": "La compra incumple al menos una regla (ej. producto en oferta o fuera de plazo).",
            "escalar_humano": "Información insuficiente en el contexto para tomar una decisión."
        }
    )
}

# Ejecución en nimble
with TypeSafeClient() as client:
    res = client.system_one(state=state_data, questions=questions)

print("¿Plazo válido?:", res.nouls["cumple_politica_tiempo"].noul > 0.5) # True
print("¿Elegible?:", res.nouls["es_producto_elegible"].noul > 0.5)        # False (es en oferta)
print("Decisión final:", res.choices["decision_final"].choice)             # 'rechazar'

¿Plazo válido?: True
¿Elegible?: False
Decisión final: rechazar


## Integración en LangChain con Contexto Dinámico
Puedes envolver este patrón dentro de una función o Runnable de LangChain que combine un contexto recuperado (por ejemplo, de una base de datos o RAG) con la entrada del usuario:

In [6]:
from langchain_core.runnables import RunnableLambda
from typesafe_sdk import TypeSafeClient, Choice

def evaluar_decision_con_contexto(inputs: dict) -> dict:
    contexto = inputs.get("contexto", "")
    pregunta = inputs.get("pregunta", "")
    
    # Creamos un estado unificado con el contexto
    state_eval = f"CONTEXTO DE REGLAS:\n{contexto}\n\nSOLICITUD A EVALUAR:\n{pregunta}"
    
    questions = {
        "accion": Choice(
            instructions="Basándote estrictamente en el CONTEXTO DE REGLAS, selecciona la acción adecuada.",
            criteria={
                "permitir": "La solicitud cumple con las reglas del contexto.",
                "denegar": "La solicitud viola las reglas descritas en el contexto.",
                "no_aplica": "El contexto no contiene información relevante para responder."
            }
        )
    }
    
    with TypeSafeClient() as client:
        result = client.system_one(state=state_eval, questions=questions)
        
    inputs["decision"] = result.choices["accion"].choice
    inputs["confianza"] = result.choices["accion"].confidence
    return inputs

# Runnable de LangChain
evaluador_chain = RunnableLambda(evaluar_decision_con_contexto)

# Ejemplo de uso
resultado = evaluador_chain.invoke({
    "contexto": "El servidor de base de datos solo acepta conexiones desde las IP: 192.168.1.10 y 192.168.1.11.",
    "pregunta": "Intentando conectar desde la IP 192.168.1.11..."
})

print("Decisión tomada:", resultado["decision"])  # 'denegar'

Decisión tomada: permitir
